<a href="https://colab.research.google.com/github/justamy20/scikit-learn-Cookbook/blob/main/13.%20Querying%20and%20Improving%20Models/Chapter_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13: Querying and Improving Models

<div class="alert alert-info">
<b>Chapter Overview:</b> Training a base machine learning model is rarely sufficient for production. This chapter covers the comprehensive workflow to improve and query models. We will explore Hyperparameter Optimization (Grid and Random Search), Machine Learning Pipelines to prevent data leakage, and Feature Selection algorithms to enhance model efficiency.
</div>

## 1. Exhaustive Grid Search (`GridSearchCV`)
**Theoretical Deep-Dive:**
Hyperparameters (like $C$ and $\gamma$ in SVM) dictate the learning process but cannot be learned directly from the data. **Grid Search** automates the discovery of optimal hyperparameters by performing an exhaustive search over a specified parameter grid.

It evaluates the model using **every single possible combination** of these values using Cross-Validation (CV). While it guarantees finding the absolute best combination within the provided grid, it is highly susceptible to the **Curse of Dimensionality**—computational cost grows exponentially with each added parameter.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the hyperparameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [1, 0.1, 0.01],
    'kernel': ['rbf', 'linear']
}

# Initialize GridSearchCV (cv=5 means 5-fold cross-validation)
grid_search = GridSearchCV(SVC(), param_grid, refit=True, cv=5)

print("Executing Grid Search (Exhaustive)...")
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_ * 100:.2f}%")

---
## 2. Randomized Parameter Optimization (`RandomizedSearchCV`)
**Theoretical Deep-Dive:**
When the hyperparameter space is vast, Grid Search is computationally prohibitive. **Randomized Search** solves this by sampling a fixed number of parameter settings from specified probability distributions (e.g., continuous uniform or exponential distributions).

Research demonstrates that Randomized Search is practically more efficient because, in most ML algorithms, only a small subset of hyperparameters significantly impacts performance. Random Search explores a wider variety of values across all dimensions rather than wasting time perfectly exploring unimportant ones.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import expon, reciprocal
import time

# Define continuous distributions
param_distributions = {
    'C': reciprocal(0.001, 1000),
    'gamma': expon(scale=1.0),
    'kernel': ['rbf', 'linear']
}

# Sample 15 random combinations
random_search = RandomizedSearchCV(SVC(), param_distributions, n_iter=15, cv=5, random_state=42)

start_time = time.time()
random_search.fit(X_train, y_train)
end_time = time.time()

print(f"Randomized Search completed in {end_time - start_time:.2f} seconds.")
print(f"Best Hyperparameters Found: {random_search.best_params_}")
print(f"Best CV Accuracy: {random_search.best_score_ * 100:.2f}%")

---
## 3. Machine Learning Pipelines (Preventing Data Leakage)
**Theoretical Deep-Dive:**
A common pitfall in model improvement is **Data Leakage** during Cross-Validation. If you scale or impute your data *before* running cross-validation, information from the validation fold "leaks" into the training phase (because the scaler calculated the mean/variance using the entire dataset).

To perform rigorous and mathematically sound Cross-Validation, data transformations must be computed *only* on the training folds. `scikit-learn` addresses this using `Pipeline`. A Pipeline chains multiple processing steps together. When used inside `GridSearchCV`, it ensures that preprocessing fits are executed strictly within the CV loops.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a pipeline: Step 1 (Scaler) -> Step 2 (Model)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

# To search parameters within a pipeline, use the syntax: stepname__parameter
pipe_param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf']
}

# Pass the entire pipeline into GridSearch
pipeline_grid = GridSearchCV(pipe, pipe_param_grid, cv=5)
pipeline_grid.fit(X_train, y_train)

print(f"Best Pipeline Parameters: {pipeline_grid.best_params_}")
print(f"Test Set Accuracy (Strictly no data leakage): {pipeline_grid.score(X_test, y_test) * 100:.2f}%")

---
## 4. Recursive Feature Elimination (RFE)
**Theoretical Deep-Dive:**
Improving a model isn't just about tweaking parameters; it's also about optimizing the data fed into it. **Feature Selection** reduces overfitting, removes noise, and speeds up computation.

**RFE** works by training the model, evaluating the importance of each feature (using `coef_` or `feature_importances_`), and recursively pruning the least important features until the desired number of features is reached.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# We use RandomForest because it natively provides feature importances
estimator = RandomForestClassifier(random_state=42)

# Select only the top 10 most important features from the original 30 features
selector = RFE(estimator, n_features_to_select=10, step=1)
selector = selector.fit(X_train, y_train)

print(f"Original number of features: {X_train.shape[1]}")
print(f"Reduced number of features: {np.sum(selector.support_)}")

# Let's see which features survived the elimination
features = data.feature_names
selected_features = features[selector.support_]
print("\nTop 10 Most Important Features selected by RFE:")
for i, feat in enumerate(selected_features, 1):
    print(f"{i}. {feat}")

---
### Chapter 13 Summary
*(Fulfilling Assignment Requirement 3.b: Summarize each chapter)*

In this concluding chapter, we examined the ecosystem of tools required to elevate a base model into a highly optimized, production-ready system. We transitioned from manual iterations to systematic engineering.

**Key Takeaways:**
1. **Hyperparameter Tuning:** We utilized `GridSearchCV` for exhaustive evaluation within a defined parameter space, and `RandomizedSearchCV` for efficient probabilistic sampling across vast, high-dimensional spaces. Both methods utilize Cross-Validation to ensure generalization.
2. **Pipelines:** We addressed the critical issue of *Data Leakage*. By chaining preprocessing steps (like `StandardScaler`) with the estimator inside a `Pipeline`, we guarantee that scaling parameters are learned strictly from the training folds during cross-validation, preserving the integrity of the test evaluations.
3. **Feature Selection:** We applied Recursive Feature Elimination (`RFE`) to mathematically deduce and retain only the most impactful features. This strategy mitigates the curse of dimensionality, reduces computational overhead, and often increases accuracy by eliminating noise.

Through the combination of tuning, pipelining, and feature selection, we conclude the machine learning workflow, capable of producing robust, scalable, and accurate predictive models.